# ITELv5 - SECCIÓN 3: GESTIÓN DE CONTENIDO (Vocales, Consonantes y Palabras Aisladas)

Escanea automáticamente las carpetas de dataset para registrar audios de:
- 01SAMOHL: Vocales
- 02SOUHL: Consonantes
- 03_1SL, 04_2SL, 05_3SL, 06_4SL, 07_VSL: Palabras aisladas de 1 a 5 sílabas

Genera listados y resúmenes por categoría (Healthy/Patients), tipo y etiqueta específica.

In [1]:
# ===============================================================_
# ITELv5 - SECCIÓN 3: GESTIÓN DE CONTENIDO (Vocales, Consonantes y Palabras Aisladas)
# ===============================================================_

%run "./ITELv5_01_CONFIGURACION.ipynb"
from pathlib import Path
import pandas as pd
from tqdm import tqdm
import re

print('\n🎧 INICIANDO GESTIÓN DE CONTENIDO (v5)')
print('=' * 60)

🔍 Verificando estructura de carpetas...

✅ Estructura de carpetas completa.
🔧 Dependencias comprobadas.

📁 BASE_DIR: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL
🖥️ GPU disponible: False
Rutas Principales de ITELv5:
Datos_Brutos:     C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Datos_Brutos
Datos Procesados: C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Datos_Procesados
Modelos:          C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Modelos
Reportes (CSV):   C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV
Reportes (Figuras): C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\Figuras
Logs:             C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Logs
🔍 Carpeta Healthy con 308 subrutas de vocales
🔍 Carpeta Patients con 378 subrutas de vocales
✅ Configuración ITELv5 lista.

🎧 INICIANDO GESTIÓN DE CONTENIDO (v5)


In [2]:
# ===============================================================_
# 3.1 — Definiciones de tipos y reglas de extracción
# ===============================================================_

TIPOS = {
    '01SAMOHL': {'tipo': 'vocal'},
    '02SOUHL': {'tipo': 'consonante'},
    '03_1SL': {'tipo': 'palabra', 'silabas': 1},
    '04_2SL': {'tipo': 'palabra', 'silabas': 2},
    '05_3SL': {'tipo': 'palabra', 'silabas': 3},
    '06_4SL': {'tipo': 'palabra', 'silabas': 4},
    '07_VSL': {'tipo': 'palabra', 'silabas': 5}
}

VOCALES = set(list('aeiou'))
CONSONANTES = set(list('bcdfghjklmnñpqrstvwxyz'))

def inferir_etiqueta(nombre_archivo, tipo):
    base = Path(nombre_archivo).stem.lower()  # sin extensión
    # Para vocal/consonante: intentar usar la última letra si es alfabética
    if tipo in ('vocal','consonante') and len(base) >= 1:
        ch = base[-1]
        if ch in VOCALES.union(CONSONANTES):
            return ch
    # Para palabras: token por guion bajo
    tokens = re.split(r'[_]+', base)
    return tokens[0] if tokens else base

def tipo_desde_subcarpeta(subcarpeta_nombre):
    return TIPOS.get(subcarpeta_nombre, {'tipo': 'desconocido'})

In [3]:
# ===============================================================_
# 3.2 — Escaneo del dataset
# ===============================================================_

def escanear_contenido(rutas_dict):
    registros = []
    subcarpetas_interes = list(TIPOS.keys())
    for categoria, lista_rutas in rutas_dict.items():
        # lista_rutas ahora proviene de 'Datos_Brutos/Healthy' o 'Datos_Brutos/Patients'
        for carpeta in lista_rutas:
            if carpeta.name not in subcarpetas_interes:
                continue
            info_tipo = tipo_desde_subcarpeta(carpeta.name)
            t = info_tipo['tipo']
            sil = info_tipo.get('silabas')
            for wav in carpeta.glob('*.wav'):
                nombre = wav.name
                # Regla solicitada: omitir archivos de Patients que contengan '_'
                if categoria == 'Patients' and '_' in nombre:
                    continue
                etiqueta = inferir_etiqueta(nombre, t)
                registros.append({
                    'categoria': categoria,
                    'subcarpeta': carpeta.name,
                    'tipo': t,
                    'silabas': sil,
                    'etiqueta': etiqueta,
                    'nombre': nombre,
                    'ruta': str(wav.resolve())
                })
    return pd.DataFrame(registros)

# Esta función ahora usa ITELConfig.DATA_DIR (Datos_Brutos) gracias a la Celda 1
rutas_vocales = ITELConfig.obtener_rutas_vocales() 
df_contenido = escanear_contenido(rutas_vocales)
print(f'✅ {len(df_contenido)} audios detectados en 01..07 (Healthy/Patients).')
display(df_contenido.head())

✅ 3730 audios detectados en 01..07 (Healthy/Patients).


,categoria,subcarpeta,tipo,silabas,etiqueta,nombre,ruta
0,Healthy,01SAMOHL,vocal,NaN,a,4PK1A.wav,C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audi...
1,Healthy,01SAMOHL,vocal,NaN,e,4PK1E.wav,C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audi...
2,Healthy,01SAMOHL,vocal,NaN,i,4PK1I.wav,C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audi...
3,Healthy,01SAMOHL,vocal,NaN,o,4PK1O.wav,C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audi...
4,Healthy,01SAMOHL,vocal,NaN,u,4PK1U.wav,C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audi...


In [4]:
# ===============================================================_
# 3.3 — Resúmenes por categoría, tipo, etiqueta y sílabas
# ===============================================================_

resumen_tipo = (df_contenido.groupby(['categoria','tipo']).size().reset_index(name='total'))
resumen_etiqueta = (df_contenido.groupby(['categoria','tipo','etiqueta']).size().reset_index(name='total'))
resumen_silabas = (df_contenido.groupby(['categoria','tipo','silabas']).size().reset_index(name='total'))
print('\n📦 RESUMEN POR TIPO:')
display(resumen_tipo)
print('\n📦 RESUMEN POR ETIQUETA:')
display(resumen_etiqueta.head(20))
print('\n📦 RESUMEN POR SÍLABAS:')
display(resumen_silabas)


📦 RESUMEN POR TIPO:


,categoria,tipo,total
0,Healthy,consonante,438
1,Healthy,palabra,1011
2,Healthy,vocal,231
3,Patients,consonante,540
4,Patients,palabra,1240
5,Patients,vocal,270



📦 RESUMEN POR ETIQUETA:


,categoria,tipo,etiqueta,total
0,Healthy,consonante,b,41
1,Healthy,consonante,d,44
2,Healthy,consonante,g,44
3,Healthy,consonante,h,44
4,Healthy,consonante,k,44
5,Healthy,consonante,k2,1
6,Healthy,consonante,l,44
7,Healthy,consonante,m,43
8,Healthy,consonante,p,2
9,Healthy,consonante,r,44



📦 RESUMEN POR SÍLABAS:


,categoria,tipo,silabas,total
0,Healthy,palabra,1.0,391
1,Healthy,palabra,2.0,219
2,Healthy,palabra,3.0,176
3,Healthy,palabra,4.0,132
4,Healthy,palabra,5.0,93
5,Patients,palabra,1.0,486
6,Patients,palabra,2.0,269
7,Patients,palabra,3.0,216
8,Patients,palabra,4.0,162
9,Patients,palabra,5.0,107


### 3.4 — Exportar listados y resúmenes

In [5]:
# ===============================================================_
# 3.4 — Exportar listados y resúmenes
# ===============================================================_

# Esta celda ahora usa ITELConfig.CSV_DIR (Reportes/CSV)
csv_dir = ITELConfig.CSV_DIR

# (Línea eliminada: csv_dir.mkdir(exist_ok=True))
# La existencia de la carpeta ya fue verificada por 01_CONFIGURACION

df_contenido.to_csv(csv_dir / 'gestion_contenido_v5.csv', index=False)
resumen_tipo.to_csv(csv_dir / 'resumen_contenido_tipo_v5.csv', index=False)
resumen_etiqueta.to_csv(csv_dir / 'resumen_contenido_etiqueta_v5.csv', index=False)
resumen_silabas.to_csv(csv_dir / 'resumen_contenido_silabas_v5.csv', index=False)

print('\n💾 Archivos exportados:')
print(f" - {csv_dir / 'gestion_contenido_v5.csv'} (listado completo)")
print(f" - {csv_dir / 'resumen_contenido_tipo_v5.csv'}")
print(f" - {csv_dir / 'resumen_contenido_etiqueta_v5.csv'}")
print(f" - {csv_dir / 'resumen_contenido_silabas_v5.csv'}")

print('\n✅ Sección 3 (v5) completada. Lista para preprocesamiento (Sección 4).')


💾 Archivos exportados:
 - C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV\gestion_contenido_v5.csv (listado completo)
 - C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV\resumen_contenido_tipo_v5.csv
 - C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV\resumen_contenido_etiqueta_v5.csv
 - C:\Users\USER\Documents\GitHub\ITEL-Tesis-Audio-TEL\Reportes\CSV\resumen_contenido_silabas_v5.csv

✅ Sección 3 (v5) completada. Lista para preprocesamiento (Sección 4).
